# Tetrahedral Gradient: correctness & performance comparison

This notebook:
1. Loads a tetrahedral mesh (same data as notebook 19)
2. Computes the gradient matrix with the **loop-based** implementation
3. Computes the gradient matrix with the **vectorized** implementation
4. Checks that the two are numerically identical
5. Benchmarks both to decide which one to use as default

In [1]:
import time

import gsops.backend as gs
import numpy as np

from geomfum.shape import TetrahedralMesh

## 1. Load tetrahedral mesh

In [2]:
mesh = TetrahedralMesh.from_file("../../../datasets/man02-rest.mesh")
print(f"Vertices: {mesh.n_vertices}, Tetrahedra: {mesh.n_tets}")

Vertices: 38295, Tetrahedra: 146804


## 2. Compute gradient — loop-based

In [ ]:
from geomfum.shape.shape_utils import (
    compute_tetrahedral_gradient_matrix,
    compute_tetrahedral_gradient_matrix_vectorized,
)

start = time.perf_counter()
G_loop = compute_tetrahedral_gradient_matrix(mesh.vertices, mesh.tets, mesh.tet_volumes)
t_loop = time.perf_counter() - start
print(f"Loop-based: {t_loop:.4f} s  |  shape: {G_loop.shape}")

Loop-based: 15.9584 s  |  shape: (440412, 38295)


## 3. Compute gradient — vectorized

In [4]:
start = time.perf_counter()
G_vec = compute_tetrahedral_gradient_matrix_vectorized(
    mesh.vertices, mesh.tets, mesh.tet_volumes
)
t_vec = time.perf_counter() - start
print(f"Vectorized:  {t_vec:.4f} s  |  shape: {G_vec.shape}")
print(f"\nSpeedup: {t_loop / t_vec:.1f}x")

Vectorized:  0.3151 s  |  shape: (440412, 38295)

Speedup: 50.6x


## 4. Correctness check

In [ ]:
import scipy.sparse as sp

# Compare sparse matrices directly (too large to densify)
diff = G_loop - G_vec
max_diff = np.max(np.abs(diff.data)) if diff.nnz > 0 else 0.0
max_val = np.max(np.abs(G_loop.data))
rel_diff = max_diff / (max_val + 1e-15)

print(f"Non-zeros loop: {G_loop.nnz},  vec: {G_vec.nnz}")
print(f"Max absolute difference: {max_diff:.2e}")
print(f"Max relative difference: {rel_diff:.2e}")
print(f"Matrices are numerically equal (rtol=1e-6): {rel_diff < 1e-6}")

Non-zeros loop: 1761648,  vec: 1761648
Max absolute difference: 5.32e-06
Max relative difference: 7.38e-08
Matrices are equal: False


## 5. Apply gradient to an eigenfunction

In [7]:
from geomfum.laplacian import LaplacianSpectrumFinder, TetrahedralLaplacianFinder
from geomfum.numerics.eig import ScipyEigsh

spectrum_finder = LaplacianSpectrumFinder(
    nonzero=False,
    fix_sign=False,
    laplacian_finder=TetrahedralLaplacianFinder(),
    eig_solver=ScipyEigsh(spectrum_size=20, sigma=-0.01),
)
mesh.laplacian.find_spectrum(
    spectrum_size=20, laplacian_spectrum_finder=spectrum_finder, set_as_basis=True
)
mesh.basis.use_k = 20

# Take the 3rd eigenfunction as a test signal
f = mesh.basis.vecs[:, 3]
print(f"Test function shape: {f.shape}")

C:\Users\giuli\OneDrive\Research\geomfum_proj\geomfum\geomfum\numerics\eig.py:48: UserWarning: M does not have the same type precision as A. This may adversely affect ARPACK convergence
  vals, vecs = scipy.sparse.linalg.eigsh(


Test function shape: (38295,)


In [8]:
from geomfum.operator import TetrahedralGradient

grad_op = TetrahedralGradient(mesh)
grad_f = grad_op(f)
print(f"Gradient shape: {grad_f.shape}  (expected: ({mesh.n_tets}, 3))")
print(f"Gradient norm (first 5 tets): {np.linalg.norm(grad_f[:5], axis=1)}")

Gradient shape: (146804, 3)  (expected: (146804, 3))
Gradient norm (first 5 tets): [0.00490846 0.12984102 0.18124296 0.0069701  0.1319166 ]


## 6. Benchmark summary

In [ ]:
n_runs = 5

times_loop = [t_loop]  # reuse the measurement from cell above
for _ in range(n_runs - 1):
    start = time.perf_counter()
    _ = compute_tetrahedral_gradient_matrix(mesh.vertices, mesh.tets, mesh.tet_volumes)
    times_loop.append(time.perf_counter() - start)

times_vec = []
for _ in range(n_runs):
    start = time.perf_counter()
    _ = compute_tetrahedral_gradient_matrix_vectorized(
        mesh.vertices, mesh.tets, mesh.tet_volumes
    )
    times_vec.append(time.perf_counter() - start)

mean_loop = np.mean(times_loop)
mean_vec = np.mean(times_vec)

print(f"Loop-based  — mean: {mean_loop:.4f} s  (std: {np.std(times_loop):.4f})")
print(f"Vectorized  — mean: {mean_vec:.4f} s  (std: {np.std(times_vec):.4f})")
print(f"Speedup: {mean_loop / mean_vec:.1f}x")
print(
    f"\nConclusion: vectorized version is {'faster' if mean_vec < mean_loop else 'slower'} → use as default"
)

Loop-based  — mean: 15.2158 s  (std: 0.4394)
Vectorized  — mean: 0.1880 s  (std: 0.0073)
Speedup: 80.9x

Conclusion: vectorized version is faster → use as default
